In [10]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from utils.fg_batting import fetch_splits, merge_splits

frames = fetch_splits(season=2026)
merged = merge_splits(frames)
merged.head()


,Team,PA_vs_lhp,K%_vs_lhp,BB%_vs_lhp,wOBA_vs_lhp,PA_vs_rhp,K%_vs_rhp,BB%_vs_rhp,wOBA_vs_rhp,PA_home,K%_home,BB%_home,wOBA_home,PA_away,K%_away,BB%_away,wOBA_away
0,ARI,286.0,0.178322,0.069930,0.365240,665.0,0.225564,0.070677,0.297433,498.0,0.166667,0.082329,0.330332,453.0,0.260486,0.057395,0.303765
1,ATH,301.0,0.259136,0.093023,0.279192,731.0,0.228454,0.095759,0.330415,397.0,0.201511,0.141058,0.342303,635.0,0.259843,0.066142,0.299039
2,ATL,353.0,0.175637,0.093484,0.351580,734.0,0.196185,0.080381,0.348152,525.0,0.182857,0.083810,0.352818,562.0,0.195730,0.085409,0.345935
3,BAL,265.0,0.249057,0.116981,0.317666,765.0,0.240523,0.100654,0.328418,535.0,0.220561,0.099065,0.352234,495.0,0.266667,0.111111,0.297094
4,BOS,269.0,0.219331,0.081784,0.303579,748.0,0.227273,0.090909,0.303328,465.0,0.221505,0.094624,0.277468,552.0,0.228261,0.083333,0.325307


In [11]:
def compute_pa_cross_splits(df: pd.DataFrame, shrinkage: int = 200) -> pd.DataFrame:
    """
    Estimate PA for home/away x vs_lhp/vs_rhp using proportionality.

    Assumes P(home | vs_lhp) ≈ P(home), i.e. venue and pitcher handedness
    are roughly independent. Shrinkage regularizes the home fraction toward
    0.5 — useful early in the season when PA_total is small.

    shrinkage: pseudo-count toward 0.5; higher = more regularization.
    """
    out = df[["Team"]].copy()

    pa_total = df["PA_vs_lhp"] + df["PA_vs_rhp"]
    alpha = pa_total / (pa_total + shrinkage)
    home_frac = alpha * (df["PA_home"] / pa_total) + (1 - alpha) * 0.5
    away_frac = 1 - home_frac

    out["PA_home_vs_lhp"] = (df["PA_vs_lhp"] * home_frac).round(1)
    out["PA_away_vs_lhp"] = (df["PA_vs_lhp"] * away_frac).round(1)
    out["PA_home_vs_rhp"] = (df["PA_vs_rhp"] * home_frac).round(1)
    out["PA_away_vs_rhp"] = (df["PA_vs_rhp"] * away_frac).round(1)

    return out


def compute_rate_cross_splits(df: pd.DataFrame, stat: str) -> pd.DataFrame:
    """
    Estimate cross-split rate stats using additive main-effects decomposition:

        rate_home_vs_lhp ≈ rate_home + rate_vs_lhp - rate_total

    This assumes venue and pitcher handedness effects are additive (no
    interaction term). Works for wOBA, K%, BB%.
    """
    out = df[["Team"]].copy()

    pa_total = df["PA_vs_lhp"] + df["PA_vs_rhp"]
    rate_total = (
        df["PA_vs_lhp"] * df[f"{stat}_vs_lhp"]
        + df["PA_vs_rhp"] * df[f"{stat}_vs_rhp"]
    ) / pa_total

    for venue in ("home", "away"):
        for hand in ("lhp", "rhp"):
            out[f"{stat}_{venue}_vs_{hand}"] = (
                df[f"{stat}_{venue}"] + df[f"{stat}_vs_{hand}"] - rate_total
            )

    return out


In [12]:
pa_splits   = compute_pa_cross_splits(merged)
woba_splits = compute_rate_cross_splits(merged, "wOBA")
kpct_splits = compute_rate_cross_splits(merged, "K%")
bbpct_splits = compute_rate_cross_splits(merged, "BB%")

result = (
    pa_splits
    .merge(woba_splits,  on="Team")
    .merge(kpct_splits,  on="Team")
    .merge(bbpct_splits, on="Team")
)

result.sort_values("Team").reset_index(drop=True)


,Team,PA_home_vs_lhp,PA_away_vs_lhp,PA_home_vs_rhp,PA_away_vs_rhp,wOBA_home_vs_lhp,wOBA_home_vs_rhp,wOBA_away_vs_lhp,wOBA_away_vs_rhp,K%_home_vs_lhp,K%_home_vs_rhp,K%_away_vs_lhp,K%_away_vs_rhp,BB%_home_vs_lhp,BB%_home_vs_rhp,BB%_away_vs_lhp,BB%_away_vs_rhp
0,ARI,148.6,137.4,345.5,319.5,0.377747,0.309941,0.351179,0.283373,0.133632,0.180874,0.227451,0.274693,0.081807,0.082554,0.056873,0.057620
1,ATH,121.4,179.6,294.9,436.1,0.306020,0.357243,0.262756,0.313979,0.223244,0.192562,0.281576,0.250894,0.139120,0.141856,0.064204,0.066940
2,ATL,171.4,181.6,356.4,377.6,0.355132,0.351705,0.348249,0.344822,0.168982,0.189530,0.181855,0.202402,0.092657,0.079554,0.094257,0.081154
3,BAL,136.8,128.2,394.9,370.1,0.344248,0.355000,0.289108,0.299860,0.226899,0.218365,0.273005,0.264471,0.111192,0.094865,0.123238,0.106910
4,BOS,124.9,144.1,347.3,400.7,0.277653,0.277402,0.325492,0.325241,0.215664,0.223606,0.222420,0.230362,0.087912,0.097037,0.076622,0.085747
5,CHC,221.7,159.3,405.0,291.0,0.383261,0.336973,0.381394,0.335106,0.194654,0.189450,0.252211,0.247007,0.134217,0.123865,0.094247,0.083896
6,CHW,116.6,161.4,317.5,439.5,0.323424,0.289737,0.350445,0.316758,0.224597,0.211901,0.270227,0.257530,0.117194,0.117236,0.095593,0.095635
7,CIN,125.2,120.8,397.0,383.0,0.344893,0.321933,0.305825,0.282865,0.271852,0.263190,0.239653,0.230991,0.154863,0.144450,0.087597,0.077185
8,CLE,130.0,150.0,352.1,405.9,0.337558,0.322307,0.319699,0.304447,0.170657,0.213722,0.185248,0.228313,0.110592,0.132124,0.071310,0.092842
9,COL,102.0,114.0,376.5,420.5,0.314564,0.352265,0.270589,0.308291,0.240568,0.206203,0.310371,0.276006,0.051024,0.082909,0.050840,0.082725
